Here is how it works:
- Shallow Clone from all branches and Config files: .yml, yaml and related .json, .sh
- The url list is sorted and locked for random sampling or stop/run to continue incrementally 
- START_NUMBER holds the last reviewed url's index
- The sample repos are stored in "Cloned_Sample"
- This will save the sample repos as well as metrics, configs, builds and test lines
- metadata includes the key metrics and project name


In [1]:
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import time, datetime
import stat
import git

# === CONFIGURATION ===
MAX_PROJECTS = 1976
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"F:\Android_Mobile_App\AndroidProjects")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "8.2-Project_Metadata.csv"
config_location_csv = base_dir / "Config_Location.csv"
config_branch_info_csv = base_dir / "Config_File_Branch_Info.csv"

# === ENSURE FOLDERS EXIST ===
for path in [clone_dir, cloned_sample_dir, yml_output_dir, build_info_dir]:
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df['project_name'] = df['github_url'].apply(lambda x: x.rstrip("/").split("/")[-1].replace(".git", ""))
df = df.sort_values(by='github_url').reset_index(drop=True)

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)

# === FUNCTION TO HANDLE READ-ONLY FILES ===
def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

# === FUNCTION TO GET COUNT FROM GITHUB API ===
def get_count(api_url, headers):
    try:
        r = requests.get(api_url, headers=headers, params={"per_page": 1})
        if 'Link' in r.headers:
            return int(r.headers['Link'].split(',')[0].split('page=')[-1].split('>')[0])
        return len(r.json())
    except:
        return 0

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.loc[i, 'github_url']
    project_name = df.loc[i, 'project_name']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', '--depth', '1', '--no-single-branch', url, str(repo_path)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("✅ Shallow clone complete")
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout while cloning {repo_name}, skipping...")
        continue

    # === Save repo to Cloned_Sample if in sample ===
    if i in sample_indices_to_keep:
        dest_sample_path = cloned_sample_dir / repo_name
        if not dest_sample_path.exists():
            try:
                shutil.copytree(repo_path, dest_sample_path)
                print(f"📁 Copied {repo_name} to Cloned_Sample")
            except Exception as e:
                print(f"⚠️ Failed to copy {repo_name} to Cloned_Sample: {e}")

    # === Extract relevant config files and branch info ===
    config_file_records = []
    extracted_count = 0
    for root, _, files in os.walk(repo_path):
        for file in files:
            lower_file = file.lower()
            if lower_file.endswith(('.yml', '.yaml', '.sh', '.json')):
                file_path = Path(root) / file
                rel_path = file_path.relative_to(repo_path)
                rel_str = str(rel_path).lower()

                if lower_file.endswith(('.sh', '.json')) and not any(x in rel_str for x in ['test', 'instrument', 'ci']):
                    continue

                try:
                    repo = git.Repo(str(repo_path))
                    commits_touching = list(repo.iter_commits(paths=str(rel_path), max_count=1))
                    branch_name = "unknown"
                    if commits_touching:
                        commit = commits_touching[0]
                        for branch in repo.branches:
                            if commit in repo.iter_commits(branch):
                                branch_name = branch.name
                                break
                except Exception:
                    branch_name = "unknown"

                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(file_path, yml_output_dir / safe_name)
                extracted_count += 1

                config_file_records.append({
                    "repo_name": repo_name,
                    "project_name": project_name,
                    "file_path": str(rel_path),
                    "file_name": file,
                    "branch": branch_name
                })

    if config_file_records:
        config_record_df = pd.DataFrame(config_file_records)
        if config_branch_info_csv.exists():
            config_record_df.to_csv(config_branch_info_csv, mode='a', header=False, index=False)
        else:
            config_record_df.to_csv(config_branch_info_csv, mode='w', header=True, index=False)

    print(f"📄 Extracted {extracted_count} config/test files with branch info")

    # === Fetch and save metadata ===
    try:
        headers = {'Authorization': f'token {GITHUB_TOKEN}'}
        base_api = f"https://api.github.com/repos/{username}/{project}"
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json()

        metadata_row = {
            'project_name': project_name,
            'repo_name': repo_name,
            'full_name': data.get('full_name'),
            'description': data.get('description'),
            'language': data.get('language'),
            'license': data.get('license', {}).get('name') if data.get('license') else None,
            'created_at': data.get('created_at'),
            'updated_at': data.get('updated_at'),
            'last_commit_date': data.get('pushed_at'),
            'stars': data.get('stargazers_count'),
            'forks': data.get('forks_count'),
            'watchers': data.get('watchers_count'),
            'open_issues': data.get('open_issues_count'),
            'contributors': get_count(f"{base_api}/contributors", headers),
            'pull_requests': get_count(f"{base_api}/pulls?state=all", headers),
            'commits': get_count(f"{base_api}/commits", headers),
            'size': data.get('size')
        }

        metadata_df = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            metadata_df.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            metadata_df.to_csv(metadata_path, mode='w', header=True, index=False)

        print("🧾 Metadata saved")

    except Exception as e:
        print(f"⚠️ Metadata error for {repo_name}: {e}")

    # === Delete cloned repo to save time ===
    try:
        shutil.rmtree(repo_path, onerror=force_remove_readonly)
        print(f"🧹 Deleted cloned repo: {repo_name}")
    except Exception as e:
        print(f"⚠️ Failed to delete repo {repo_name}: {e}")

    # === Save updated START_NUMBER ===
    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

print("\n🏁 Finished processing selected projects.")



🔍 [1/1796] Processing 0000.01sadra.Detoxiom...
✅ Shallow clone complete
📄 Extracted 1 config/test files with branch info
🧾 Metadata saved
🧹 Deleted cloned repo: 0000.01sadra.Detoxiom

🔍 [2/1796] Processing 0001.0Kirby.ProgressNote...
✅ Shallow clone complete
📄 Extracted 2 config/test files with branch info
🧾 Metadata saved
🧹 Deleted cloned repo: 0001.0Kirby.ProgressNote

🔍 [3/1796] Processing 0002.0xf104a.NextcloudServices...
✅ Shallow clone complete
📄 Extracted 1 config/test files with branch info
🧾 Metadata saved
🧹 Deleted cloned repo: 0002.0xf104a.NextcloudServices

🔍 [4/1796] Processing 0003.0xpr03.VocableTrainer-Android...
✅ Shallow clone complete
📄 Extracted 4 config/test files with branch info
🧾 Metadata saved
🧹 Deleted cloned repo: 0003.0xpr03.VocableTrainer-Android

🔍 [5/1796] Processing 0004.20Nick.OPMT...
✅ Shallow clone complete
📄 Extracted 1 config/test files with branch info
🧾 Metadata saved
🧹 Deleted cloned repo: 0004.20Nick.OPMT

🔍 [6/1796] Processing 0005.2439905184.o

KeyboardInterrupt: 